In [0]:
# Load the source data into a DataFrame from a SQL table in the user's database
taxi_df = spark.sql("SELECT * FROM users.jamin_solensky.taxi")

# Transformation: Filter the DataFrame to include only trips with a distance greater than 2 miles
filtered_df = taxi_df.filter(taxi_df.trip_distance > 2)

# Import necessary functions for timestamp conversion and column operations
from pyspark.sql.functions import unix_timestamp, col

# Transformation: Add a new column to the DataFrame that calculates the trip duration in minutes
# This is done by subtracting the pickup timestamp from the dropoff timestamp and converting the result from seconds to minutes
transformed_df = filtered_df.withColumn(
    "trip_duration_minutes",
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60
)

# Aggregation: Group the transformed DataFrame by the pickup zip code
# Calculate the average fare amount and average trip duration for each group
# Rename the resulting columns for clarity
aggregated_df = transformed_df.groupBy("pickup_zip").agg(
    {"fare_amount": "avg", "trip_duration_minutes": "avg"}
).withColumnRenamed("avg(fare_amount)", "avg_fare_amount") \
 .withColumnRenamed("avg(trip_duration_minutes)", "avg_trip_duration_minutes")

# Display the final aggregated DataFrame to visualize the results
display(aggregated_df)